# 1) About

## Purpose

This notebook establishes a **rule-based baseline** for AML fraud detection before building ML models. Rules provide a simple, interpretable floor that ML models must beat to justify their complexity.

## Connection to Feature Engineering
We will use the dataset we have produced in feature engineering notebook. 

- 29,997 rows (9999 accounts x 3 monthly snapshots)
- 28 behavioral features
- ~2.9% positive label rate

## Approach

1. Time based train/test split (Train: Apr, May. Test: June.)
2. Define evaluation framework (Precision@TopK, Recall@TopK)
3. Design simple rules based on EDA findings (fan-in)
4. Apply to test set and establish the baseline to beat

## Why rules first?

- **Interpretable:** Investigators can understand exactly why each account was flagged
- **Baseline for ML:** If ML doesn't beat rules, the complexity is not justified
- **Production reality:** Most banks start with rules and add ML on top.

## Why time-based split for rules?

Rule based models don't learn like ML but there are key reasons why I am moving forward with time based split:

1. Threshold Selection: Even though rules don't train, we pick thresholds looking at training data distributions. If I choose thresholds using the test set, result would artifically look good. Kind of data leakage.
2. Fair Comparison: Since we are going to compare ML vs. Rule Based approach later, both must be evaluated on the same held-out dataset.

-> Pick thresolds using training data, evaluate only on test data.


# 2) Imports and Data Loading

In [154]:

import pandas as pd, numpy as np
from scipy import stats
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.float_format', '{:.2f}'.format)

# Load the data
data_path = Path("../data/processed/modeling_dataset.csv")
df = pd.read_csv(data_path)
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'])

# Inspect
print(f"Shape: {df.shape}")
print(f"\nSnapshots: {df['snapshot_date'].unique()}")
print(f"\nLabel Disribution: \n{df['label'].value_counts()}")
print(f"\nPositive Label Rate: {df['label'].mean():.1%}")


Shape: (29997, 31)

Snapshots: <DatetimeArray>
['2020-04-01 00:00:00', '2020-05-01 00:00:00', '2020-06-01 00:00:00']
Length: 3, dtype: datetime64[us]

Label Disribution: 
label
0    29128
1      869
Name: count, dtype: int64

Positive Label Rate: 2.9%


# 3) Train-Test Split (Time Based)

We will split data by snapshot date.

-> Train: April + May Snapshots - to pick the thresholds

-> Test: June snapshot - for final evaluation

This split stimulates production.

In [156]:
train_df = df[df['snapshot_date'] < '2020-06-01'].copy()
test_df = df[df['snapshot_date'] >= '2020-06-01'].copy()

print(f"Train: {len(train_df):,} rows | {train_df['label'].sum()} fraudulent clients | {train_df['label'].mean()*100:.2f}% fraudulent rate")
print(f"Test:  {len(test_df):,} rows | {test_df['label'].sum()} fraud ({test_df['label'].mean()*100:.2f}%)")


Train: 19,998 rows | 527 fraudulent clients | 2.64% fraudulent rate
Test:  9,999 rows | 342 fraud (3.42%)


# 4) Evaluation Framework

Let's define the evaluation frameworks before writing the rules.

Rules produce **binary flags** not probabilities. Each account is either alerted or not. So the right questions are: 

- From the accounts we flag, how many are true positives? (Precision)
- And of all the positive cases, how many do we catch? (Recall)

**Metrics:**

- `precision` from the accounts we alert, what % are true positives? (Workload quality)
- `recall` of all Positive cases, what % do we catch? (coverage)

In [157]:

def evaluate_rule(flags, ground_truth_labels, rule_name = 'rule'):
    """
    Evaluate a binary rule flag against ground truth labels we have created in feature engineering nb (`label`)
    Params
    ------
    flags               : array-like binary rule output (1: alert, 0: not)
    ground_truth_labels : array-like ground truth binary labels (1: Positive)
    rule_name           : str - name of the rule
    """

    # flags coming from rules
    flags = np.array(flags)
    # labels coming from my labels created in feature engineering
    labels = np.array(ground_truth_labels)

    tp = int(((flags == 1) & (labels == 1)).sum())
    fp = int(((flags == 1) & (labels == 0)).sum())
    fn = int(((flags == 0) & (labels == 1)).sum())
    number_alerts = tp + fp
    total_positives = tp + fn

    precision = tp / number_alerts if number_alerts > 0 else 0.0
    recall = tp / total_positives if number_alerts > 0 else 0.0

    print(f"\n{rule_name}")
    # print(f"\n Alerts created: {number_alerts:,}, {number_alerts/len(flags)*100:.1f}% of all accounts")
    print(f"Precision: {precision:.2%}")
    print(f"Recall: {recall:.2%} ")
    print(f" TP = {tp} ||| FP = {fp} ||| FN = {fn}")
    
    #return dict(rule=rule_name, number_alerts=number_alerts, precision=round(precision, 2), recall = round(recall, 2),
    #            tp=tp, fp=fp, fn=fn)
    # or df
    return pd.DataFrame([dict(rule=rule_name, number_alerts=number_alerts, precision=round(precision, 2), recall = round(recall, 2),
               tp=tp, fp=fp, fn=fn)])


# example scenario
my_flags = [0, 1, 0, 1]
my_labels = [1, 1, 1, 1]

evaluate_rule(my_flags, my_labels, 'thisisarule')



thisisarule
Precision: 100.00%
Recall: 50.00% 
 TP = 2 ||| FP = 0 ||| FN = 2


,rule,number_alerts,precision,recall,tp,fp,fn
0,thisisarule,2,1.00,0.50,2,0,2


# 5) Rule Design

Rules are based on the strongest fan-in signals identified in EDA. Thresholds are set from **training data only** — never from test data, to avoid leakage.

**Rule 1:** `recent_unique_senders > Nth percentile` — accounts receiving from many distinct senders 

**Rule 2:** `recent_recv_sent_ratio > Nth percentile` — accounts that receive much more than they send

**Rule 3:** Rule 1 OR Rule 2 — combined coverage

**Threshold strategy:** Simulate the real-life champion-challenger process used in production AML:

1. Sample the top 100 accounts from the high-risk tail of each feature (May snapshot — point-in-time, no account duplication)
2. Check their labels — simulating analyst verdicts, but using labels we created
3. Find the lowest feature value where fraud appears in the sample
4. Find what percentile that value sits at in the full May population
5. That percentile **is** the threshold — portable and analyst-informed (From this percentile, I capture top risk clients with potential fraud)

This mirrors how real AML rules are expressed in production: *"flag accounts above the Nth percentile"* rather than a fixed value tied to one dataset. When applied to new data, the percentile is recomputed on that dataset — so the rule self-adjusts.

## a. Setting the threshold

In [175]:
SAMPLE_SIZE = 100

# Use only the most recent training snapshot (May)
# point-in-time, no account duplication across snapshots
sample_may = train_df[train_df['snapshot_date'] == '2020-05-01']

######### Rule 1: recent_unique_senders #########
# Step 1: take top 100 accounts by feature value (high-risk tail)
r1_sample = sample_may.nlargest(SAMPLE_SIZE, 'recent_unique_senders')[['ACCOUNT_ID', 'recent_unique_senders', 'label']]

# Step 2: find the lowest value where fraud appears in the sample
r1_fraud_min = r1_sample[r1_sample['label'] == 1]['recent_unique_senders'].min()

# Step 3: find what percentile that value sits at in the full May population
# this percentile IS the threshold — portable and analyst-informed
r1_threshold_pct = round(stats.percentileofscore(sample_may['recent_unique_senders'], r1_fraud_min),1)

print("Rule 1 — recent_unique_senders")
print(f"  Fraud accounts in sample : {r1_sample['label'].sum()} / {SAMPLE_SIZE}")
print(f"  Lowest fraud value       : {r1_fraud_min:.2f}")
print(f"  Threshold                : above the {r1_threshold_pct}th percentile")

######### Rule 2: recent_recv_sent_ratio #########

# Step 1: take top 100 accounts by feature value (high-risk tail)
r2_sample = sample_may.nlargest(SAMPLE_SIZE, 'recent_recv_sent_ratio')[['ACCOUNT_ID', 'recent_recv_sent_ratio', 'label']]

# Step 2: find the lowest value where fraud appears in the sample
r2_fraud_min = r2_sample[r2_sample['label'] == 1]['recent_recv_sent_ratio'].min()

# Step 3: find what percentile that value sits at in the full May population
# this percentile IS the threshold — portable and analyst-informed
r2_threshold_pct = round(stats.percentileofscore(sample_may['recent_recv_sent_ratio'], r2_fraud_min),1)

print("\nRule 2 — recent_recv_sent_ratio")
print(f"  Fraud accounts in sample : {r2_sample['label'].sum()} / {SAMPLE_SIZE}")
print(f"  Lowest fraud value       : {r2_fraud_min:.2f}")
print(f"  Threshold                : above the {r2_threshold_pct}th percentile")


Rule 1 — recent_unique_senders
  Fraud accounts in sample : 24 / 100
  Lowest fraud value       : 30.00
  Threshold                : above the 99.1th percentile

Rule 2 — recent_recv_sent_ratio
  Fraud accounts in sample : 4 / 100
  Lowest fraud value       : 12.41
  Threshold                : above the 99.2th percentile


# 6) Apply Rules to Test Set

Apply each rule to the test set (June snapshot) using the percentile thresholds learned from the analyst sample.

Both rules' threshold is determined as 99th percentile. (Top 1% for each feature)

Each account is flagged if its feature value exceeds the threshold percentile cutoff.

Results are evaluated using `evaluate_rule()` defined in Section 4.

In [185]:
# Compute cutoff values for the june based on the learned percentiles from the test set
r1_cutoff = round(np.percentile(test_df['recent_unique_senders'],  r1_threshold_pct),2)
r2_cutoff = round(np.percentile(test_df['recent_recv_sent_ratio'], r2_threshold_pct),2)
print(f"rule 1  ->  recent_unique_senders > {r1_cutoff} \nrule 2 -> recent_recv_sent_ratio > {r2_cutoff}")

# # Apply rules
test_df['r1_flag'] = (test_df['recent_unique_senders']  > r1_cutoff).astype(int)
test_df['r2_flag'] = (test_df['recent_recv_sent_ratio'] > r2_cutoff).astype(int)
test_df['r3_flag'] = ((test_df['r1_flag'] == 1) | (test_df['r2_flag'] == 1)).astype(int)


rule 1  ->  recent_unique_senders > 32.0 
rule 2 -> recent_recv_sent_ratio > 12.99


## Results

In [198]:
print(test_df[['ACCOUNT_ID', 'label', 'r1_flag', 'r2_flag', 'r3_flag']].head())

print(f"{'_'*50}")

print('\nCheck flagged percentages for each rule:')
for rule_flag in ['r1_flag', 'r2_flag', 'r3_flag']:
    print(f"\n{test_df[rule_flag].value_counts()}")
    print(f"{test_df[rule_flag].sum() / test_df.size:.2%}")


       ACCOUNT_ID  label  r1_flag  r2_flag  r3_flag
19998           1      0        0        0        0
19999           2      0        0        0        0
20000           3      0        0        0        0
20001           4      0        0        0        0
20002           5      0        0        0        0
__________________________________________________

Check flagged percentages for each rule:

r1_flag
0    9910
1      89
Name: count, dtype: int64
0.03%

r2_flag
0    9919
1      80
Name: count, dtype: int64
0.02%

r3_flag
0    9832
1     167
Name: count, dtype: int64
0.05%


## Evaluation

In [201]:
# Evaluate
r1_results = evaluate_rule(test_df['r1_flag'], test_df['label'], 'Rule 1 — unique senders')
r2_results = evaluate_rule(test_df['r2_flag'], test_df['label'], 'Rule 2 — recv sent ratio')
r3_results = evaluate_rule(test_df['r3_flag'], test_df['label'], 'Rule 3 — combined')

# Comparison table
summary = pd.concat([r1_results, r2_results, r3_results]).reset_index(drop=True)
print("\n── Rule Comparison ──")
print()
print(summary.to_string(index=False))



Rule 1 — unique senders
Precision: 25.84%
Recall: 6.73% 
 TP = 23 ||| FP = 66 ||| FN = 319

Rule 2 — recv sent ratio
Precision: 2.50%
Recall: 0.58% 
 TP = 2 ||| FP = 78 ||| FN = 340

Rule 3 — combined
Precision: 14.97%
Recall: 7.31% 
 TP = 25 ||| FP = 142 ||| FN = 317

── Rule Comparison ──

                RuleName  number_alerts  precision  recall  tp  fp  fn
 Rule 1 — unique senders             89       0.26    0.07  23  66 319
Rule 2 — recv sent ratio             80       0.03    0.01   2  78 340
       Rule 3 — combined            167       0.15    0.07  25 142 317


# 7) Summary

## Results

|Rule |Alerts | Precision | Recall | TP | FP | FN |
|---|---|---|---|---|---|---|
| Rule 1 — unique senders   | 89  | 26% | 7% | 23 | 66  | 319 |
| Rule 2 — recv sent ratio  | 77  | 3%  | 1% | 2  | 75  | 340 |
| Rule 3 — combined OR      | 164 | 15% | 7% | 25 | 139 | 317 |

## Key Findings
- **Rule 1 is the strongest baseline** — 26% precision at 7% recall. Simple, interpretable, and grounded in EDA findings.
- **Rule 2 is a weak signal** — nearly random performance. The `recv_sent_ratio` feature has extreme outliers that dominate the 99th percentile threshold, making it unreliable as a standalone rule.
- **Rule 3 (OR) does not help** — combining Rule 1 and Rule 2 adds 75 false positives while catching only 2 additional fraud accounts. The OR combination is not worth the noise.

## Baseline to Beat
ML models will be evaluated against Rule 1:
- **Precision: 26%**
- **Recall: 7%**

## Limitations
- **Low recall by design** — the 99th percentile threshold is very conservative. A lower threshold would catch more fraud but at the cost of precision.
- **Fixed rules, no learning** — thresholds are set once from the analyst sample and never adapt to new patterns.
- **Mixed labels** — positive labels include both fan-in and cycle SAR accounts. Rules are fan-in focused, so cycle fraud accounts are largely missed. A natural future iteration would train separate models per pattern type with pattern-specific labels.
- **Synthetic data** — AMLSim has zero false positives (all alerted accounts are IS_FRAUD=1). In production, the label noise would be higher and precision would likely be lower.

## Next Steps
→ See `04_modeling.ipynb`:
1. **Logistic Regression** — interpretable probabilistic model, compare precision/recall vs Rule 1
2. **LightGBM** — non-linear model to capture feature interactions
3. **SHAP** — explainability layer to understand which features drive risk scores